# Análise de Desempenho de Indicadores

## Objetivo
Automatizar a avaliação de indicadores por meio da comparação entre resultados observados e benchmarks.

## Problema
A análise manual dificultava a identificação de indicadores críticos, tendências e oportunidades de melhoria.

## Solução
Este notebook realiza:

- limpeza e padronização dos dados;
- comparação entre resultado e benchmark;
- classificação automática dos indicadores;
- cálculo de score;
- cálculo da variação entre anos;
- visualização da distribuição dos indicadores.

## Tecnologias
- Python
- Pandas
- NumPy
- Matplotlib

> **Observação:** Os dados originais foram substituídos por uma estrutura genérica para preservar informações confidenciais.


In [ ]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# =============================================================================
# CONFIGURAÇÕES
# =============================================================================

ARQUIVO_DADOS = Path("dados/dados_indicadores.xlsx")

ANOS = [2023, 2024, 2025]
TOLERANCIA = 0.05

MAPA_SCORE = {
    "Acima do esperado": 1,
    "Dentro do esperado": 0,
    "Abaixo do esperado": -1,
}

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(message)s"
)


In [ ]:
# =============================================================================
# LEITURA DOS DADOS
# =============================================================================

logging.info("Carregando base de indicadores...")

df = pd.read_excel(ARQUIVO_DADOS)

logging.info("%s indicadores carregados.", len(df))


In [ ]:
# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def converter_numero(valor):
    """Converte valores percentuais ou numéricos para float."""

    if pd.isna(valor):
        return np.nan

    valor = (
        str(valor)
        .strip()
        .replace("%", "")
        .replace(".", "")
        .replace(",", ".")
    )

    try:
        return float(valor)
    except ValueError:
        return np.nan


def classificar_indicador(resultado, benchmark, classificacao):
    """Classifica o desempenho considerando a direção clínica do indicador."""

    if pd.isna(resultado) or pd.isna(benchmark):
        return "Sem dados"

    if benchmark == 0:
        diferenca = 0 if resultado == 0 else float("inf")
    else:
        diferenca = (resultado - benchmark) / abs(benchmark)

    classificacao = str(classificacao).lower()

    if "maior" in classificacao:

        if diferenca > TOLERANCIA:
            return "Acima do esperado"

        if diferenca < -TOLERANCIA:
            return "Abaixo do esperado"

        return "Dentro do esperado"

    if "menor" in classificacao:

        if diferenca < -TOLERANCIA:
            return "Acima do esperado"

        if diferenca > TOLERANCIA:
            return "Abaixo do esperado"

        return "Dentro do esperado"

    return "Sem classificação"


def variacao_relativa(atual, anterior):
    """Calcula a variação relativa entre dois períodos."""

    if pd.isna(atual) or pd.isna(anterior):
        return np.nan

    if anterior == 0:
        return np.nan

    return (atual - anterior) / abs(anterior)


In [ ]:
# =============================================================================
# PADRONIZAÇÃO DOS DADOS
# =============================================================================

for ano in ANOS:
    df[f"Resultado_{ano}"] = df[f"Resultado_{ano}"].apply(converter_numero)
    df[f"Benchmark_{ano}"] = df[f"Benchmark_{ano}"].apply(converter_numero)


In [ ]:
# =============================================================================
# CLASSIFICAÇÃO DOS INDICADORES
# =============================================================================

# A comparação considera automaticamente se o indicador é do tipo
# 'Quanto maior, melhor' ou 'Quanto menor, melhor'.

for ano in ANOS:

    df[f"Status_{ano}"] = df.apply(
        lambda linha: classificar_indicador(
            linha[f"Resultado_{ano}"],
            linha[f"Benchmark_{ano}"],
            linha["Classificacao"],
        ),
        axis=1,
    )

    df[f"Score_{ano}"] = df[f"Status_{ano}"].map(MAPA_SCORE)


In [ ]:
# =============================================================================
# VARIAÇÃO ENTRE ANOS
# =============================================================================

df["Variacao_2025"] = df.apply(
    lambda linha: variacao_relativa(
        linha["Resultado_2025"],
        linha["Resultado_2024"],
    ),
    axis=1,
)


In [ ]:
# =============================================================================
# VISUALIZAÇÃO
# =============================================================================

status = (
    df["Status_2025"]
    .value_counts()
    .reindex(
        [
            "Acima do esperado",
            "Dentro do esperado",
            "Abaixo do esperado",
        ],
        fill_value=0,
    )
)

ax = status.plot(
    kind="bar",
    figsize=(7, 4),
    rot=0,
)

ax.set_title("Distribuição dos Indicadores por Classificação")
ax.set_xlabel("")
ax.set_ylabel("Quantidade")

plt.tight_layout()
plt.show()
